# Contract clause extraction: fine-tune Qwen3.5-4B, Gemma 4 E4B, and Gemma 4 E2B

One notebook, three fine-tunes on the same 200-row golden dataset used for this task's GEPA prompt-optimization runs, evaluated with the **same metric on the same held-out split**, so results are directly comparable across model families and sizes.

The teacher scores baked into the comparison cells come from the published runs in this task's README (GLM 5.2: 34.67 baseline, optimized blocked by OpenCode gateway instability; gpt-5.4-mini: 34.00 baseline, 36.33 GEPA-optimized -- the primary comparison, since GLM's optimize run never completed). This notebook does NOT re-run GEPA -- only the three fine-tunes below are new measurements.

**Go criterion:** each fine-tuned SLM matches or beats gpt-5.4-mini's plain-prompt baseline (34.00).

Runtime: Colab **T4 (free)** works for each model in 4-bit; A100 (Colab Pro) is faster. Runtime -> Change runtime type -> GPU. Models are loaded and fine-tuned one at a time, with GPU memory freed between each, so all three fit in one T4 session.

In [ ]:
# 1. Install (Colab)
# Pinned: unsloth/unsloth-zoo 2026.7.1 (released 2026-07-07) added a new XET-based
# downloader (unsloth_zoo/hf_xet_fallback.py) that fails on Colab with
# DownloadStallError even when HF_HUB_DISABLE_XET=1 is set. These are the last
# versions before that change -- the same ones the receipt-extraction notebooks
# ran successfully on.
!pip install -q unsloth==2026.6.9 unsloth-zoo==2026.6.7

In [ ]:
# 2. Fetch the same golden dataset (no upload needed) and replicate the GEPA run's exact split
import io
import json
import random
import urllib.request
import zipfile

SEED = 42
ROW_COUNT = 200
SOURCE_URL = "https://github.com/TheAtticusProject/cuad/raw/main/data.zip"

CATEGORIES = ["Document Name", "Parties", "Agreement Date", "Governing Law", "Anti-Assignment"]
FIELD_KEYS = {
    "Document Name": "document_name",
    "Parties": "parties",
    "Agreement Date": "agreement_date",
    "Governing Law": "governing_law",
    "Anti-Assignment": "anti_assignment",
}
WINDOW = 350

def extract_fields(paragraph):
    found = {}
    for qa in paragraph["qas"]:
        for cat in CATEGORIES:
            if f'"{cat}"' in qa["question"] and qa["answers"]:
                ans = qa["answers"][0]
                found[cat] = (ans["text"].strip(), ans["answer_start"])
    if "Document Name" not in found or "Parties" not in found:
        return None
    return found

def build_row(context, found):
    spans = sorted(found.items(), key=lambda kv: kv[1][1])
    windows = []
    for _cat, (text, start) in spans:
        lo = max(0, start - WINDOW)
        hi = min(len(context), start + len(text) + WINDOW)
        windows.append(context[lo:hi].strip())
    excerpt = "\n\n".join(windows)
    output = {FIELD_KEYS[cat]: found[cat][0] if cat in found else "" for cat in CATEGORIES}
    input_text = (
        "Extract contract fields as a JSON object with keys document_name, parties, "
        "agreement_date, governing_law, anti_assignment. Use \"\" for any field not "
        "present in the excerpt.\n\n"
        "Contract excerpt:\n" + excerpt
    )
    return {"text": input_text, "expected": json.dumps(output, ensure_ascii=False, sort_keys=True)}

with urllib.request.urlopen(SOURCE_URL, timeout=120) as resp:
    raw = resp.read()
with zipfile.ZipFile(io.BytesIO(raw)) as zf:
    with zf.open("CUADv1.json") as f:
        data = json.load(f)["data"]

rows = []
for entry in data:
    for paragraph in entry["paragraphs"]:
        found = extract_fields(paragraph)
        if found is None:
            continue
        rows.append(build_row(paragraph["context"], found))

if len(rows) < ROW_COUNT:
    raise RuntimeError(f"expected at least {ROW_COUNT} rows, got {len(rows)}")

# CRITICAL: same sample seed as prepare_data.py in this task folder, so the rows
# (and therefore the held-out split after this notebook's own shuffle below) are
# identical across every model this task is fine-tuned on.
rows = random.Random(SEED).sample(rows, ROW_COUNT)
random.Random(SEED).shuffle(rows)
split = int(len(rows) * 0.7)
trainset, devset = rows[:split], rows[split:]
print(f"train {len(trainset)}, held-out {len(devset)}")

In [ ]:
# 3. Same Tier-1 metric as the GEPA spike (pure-python copy of json_field_metric)
def json_field_f1(expected_str: str, actual_str: str) -> float:
    try:
        expected = json.loads(expected_str)
        actual = json.loads(actual_str)
    except json.JSONDecodeError:
        return 0.0
    if not isinstance(expected, dict) or not isinstance(actual, dict):
        return 0.0
    tp = sum(1 for k, v in expected.items() if actual.get(k) == v)
    precision = tp / len(actual) if actual else 0.0
    recall = tp / len(expected) if expected else 0.0
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0

In [ ]:
# 4. Eval + train helpers, shared by all three models below (identical logic,
# defined once rather than three times)
import re

def extract_json(s: str) -> str:
    s = re.sub(r"<think>.*?(</think>|$)", "", s, flags=re.DOTALL)
    s = re.sub(r"```(json)?", "", s)
    m = re.search(r"\{.*\}", s, re.DOTALL)
    return m.group(0) if m else s

def evaluate_slm(model, tokenizer, devset) -> float:
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)
    scores = []
    for row in devset:
        messages = [{"role": "user", "content": row["text"]}]
        try:
            prompt = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, tokenize=False, enable_thinking=False
            )
        except TypeError:
            prompt = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, tokenize=False
            )
        # text= keyword, not a positional arg: these models ship a multimodal
        # Processor, and a positional arg is interpreted as an image.
        inputs = tokenizer(text=prompt, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=512, do_sample=False)
        completion = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        scores.append(json_field_f1(row["expected"], extract_json(completion)))
    return 100 * sum(scores) / len(scores)

def to_text(tokenizer, row):
    messages = [
        {"role": "user", "content": row["text"]},
        {"role": "assistant", "content": row["expected"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

def free_gpu_memory():
    import gc
    import torch
    gc.collect()
    torch.cuda.empty_cache()

results = {}  # each model's raw/fine-tuned scores land here as we go

## Qwen3.5-4B

`Qwen/Qwen3.5-4B`

In [ ]:
# 5. Load Qwen3.5-4B in 4-bit + LoRA
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    "Qwen/Qwen3.5-4B",
    max_seq_length=4096,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

In [ ]:
# 6. Pre-finetune baseline of the raw SLM on the held-out split
raw_slm_score = evaluate_slm(model, tokenizer, devset)
print(f"Qwen3.5-4B raw (no fine-tune): {raw_slm_score:.2f}")

In [ ]:
# 7. Build chat-format training data and fine-tune (SFT)
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

train_ds = Dataset.from_list([to_text(tokenizer, r) for r in trainset])

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=5,
        output_dir="outputs-apprentice-qwen35-4b-lora-cuad",
        report_to="none",
    ),
)
trainer.train()

In [ ]:
# 8. Post-finetune eval on the SAME held-out split: the comparison that matters
finetuned_score = evaluate_slm(model, tokenizer, devset)
results["Qwen3.5-4B"] = {"raw": raw_slm_score, "finetuned": finetuned_score}

glm52_baseline = 34.67  # optimized blocked -- OpenCode gateway instability, see README
gpt54mini_baseline = 34.00
gpt54mini_optimized = 36.33

print("=" * 52)
print(f"glm-5.2 baseline prompt      : {glm52_baseline:.2f}")
print(f"gpt-5.4-mini baseline prompt : {gpt54mini_baseline:.2f}")
print(f"gpt-5.4-mini GEPA-optimized  : {gpt54mini_optimized:.2f}")
print(f"Qwen3.5-4B raw               : {raw_slm_score:.2f}")
print(f"Qwen3.5-4B fine-tuned        : {finetuned_score:.2f}")
print("=" * 52)
print("GO criterion: fine-tuned SLM >= gpt-5.4-mini baseline prompt score (34.00)")

In [ ]:
# 9. Persist the adapter (plan: artifacts go to a private HF Hub repo, not Drive),
# then free GPU memory before the next model
model.save_pretrained("apprentice-qwen35-4b-lora-cuad")
tokenizer.save_pretrained("apprentice-qwen35-4b-lora-cuad")
print("saved -> apprentice-qwen35-4b-lora-cuad/")

del model, tokenizer, trainer
free_gpu_memory()

In [ ]:
# 9b. Write the model card and push to your HF Hub repo (same card style as
# the published receipt-extraction adapters).
# Fill in the printed numbers from cell 8 before pushing -- never push a card
# with a number that was not actually printed by this run.
from huggingface_hub import login

login()  # paste a HF token with write access when prompted

HF_USERNAME = "YOUR_HF_USERNAME"
REPO_ID = f"{HF_USERNAME}/apprentice-qwen35-4b-lora-cuad"

model_card = """---
datasets: [theatticusproject/cuad-qa]
base_model: Qwen/Qwen3.5-4B
library_name: peft
license: apache-2.0
tags: [lora, unsloth, contract-clause-extraction, legal, apprentice]
---

# Apprentice --- Qwen3.5-4B LoRA (contract clause extraction)

LoRA adapter fine-tuned on 140 golden examples (CUAD v1, Contract Understanding
Atticus Dataset, 510 real commercial contracts from SEC EDGAR, expert-annotated
by lawyers, CC-BY-4.0) to extract document_name, parties, agreement_date,
governing_law, anti_assignment from a contract excerpt as JSON.

## Results (60 held-out rows, field-level F1)

Fill in from this task's README after publishing this run's printed numbers.

## Training

LoRA r=16, alpha 16, 3 epochs, lr 2e-4, batch 2 x grad-accum 4, Unsloth 4-bit,
Colab GPU. Train/eval split: seed 42, 140/60 from 200 sampled rows -- identical
split across every model this task is fine-tuned on.

## Usage

Load with PEFT on top of `Qwen/Qwen3.5-4B`, or serve via vLLM with `--enable-lora`.
Caveat: evaluated on 60 rows with exact-string field matching -- re-validate
before production use.
"""

with open("apprentice-qwen35-4b-lora-cuad/README.md", "w") as f:
    f.write(model_card)

model.push_to_hub(REPO_ID, private=False)
tokenizer.push_to_hub(REPO_ID, private=False)

from huggingface_hub import upload_file
upload_file(path_or_fileobj="apprentice-qwen35-4b-lora-cuad/README.md", path_in_repo="README.md", repo_id=REPO_ID)
print(f"pushed -> https://huggingface.co/{REPO_ID}")

## Gemma 4 E4B

`google/gemma-4-E4B-it` -- natively multimodal (text/image/audio in, text out) but loads and fine-tunes as a plain causal LM for this text-only task

In [ ]:
# 10. Load Gemma 4 E4B in 4-bit + LoRA
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    "google/gemma-4-E4B-it",
    max_seq_length=4096,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

In [ ]:
# 11. Pre-finetune baseline of the raw SLM on the held-out split
raw_slm_score = evaluate_slm(model, tokenizer, devset)
print(f"Gemma 4 E4B raw (no fine-tune): {raw_slm_score:.2f}")

In [ ]:
# 12. Build chat-format training data and fine-tune (SFT)
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

train_ds = Dataset.from_list([to_text(tokenizer, r) for r in trainset])

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=5,
        output_dir="outputs-apprentice-gemma4-e4b-lora-cuad",
        report_to="none",
    ),
)
trainer.train()

In [ ]:
# 13. Post-finetune eval on the SAME held-out split: the comparison that matters
finetuned_score = evaluate_slm(model, tokenizer, devset)
results["Gemma 4 E4B"] = {"raw": raw_slm_score, "finetuned": finetuned_score}

glm52_baseline = 34.67  # optimized blocked -- OpenCode gateway instability, see README
gpt54mini_baseline = 34.00
gpt54mini_optimized = 36.33

print("=" * 52)
print(f"glm-5.2 baseline prompt      : {glm52_baseline:.2f}")
print(f"gpt-5.4-mini baseline prompt : {gpt54mini_baseline:.2f}")
print(f"gpt-5.4-mini GEPA-optimized  : {gpt54mini_optimized:.2f}")
print(f"Gemma 4 E4B raw               : {raw_slm_score:.2f}")
print(f"Gemma 4 E4B fine-tuned        : {finetuned_score:.2f}")
print("=" * 52)
print("GO criterion: fine-tuned SLM >= gpt-5.4-mini baseline prompt score (34.00)")

In [ ]:
# 14. Persist the adapter (plan: artifacts go to a private HF Hub repo, not Drive),
# then free GPU memory before the next model
model.save_pretrained("apprentice-gemma4-e4b-lora-cuad")
tokenizer.save_pretrained("apprentice-gemma4-e4b-lora-cuad")
print("saved -> apprentice-gemma4-e4b-lora-cuad/")

del model, tokenizer, trainer
free_gpu_memory()

In [ ]:
# 14b. Write the model card and push to your HF Hub repo (same card style as
# the published receipt-extraction adapters).
# Fill in the printed numbers from cell 13 before pushing -- never push a card
# with a number that was not actually printed by this run.
from huggingface_hub import login

login()  # paste a HF token with write access when prompted

HF_USERNAME = "YOUR_HF_USERNAME"
REPO_ID = f"{HF_USERNAME}/apprentice-gemma4-e4b-lora-cuad"

model_card = """---
datasets: [theatticusproject/cuad-qa]
base_model: google/gemma-4-E4B-it
library_name: peft
license: apache-2.0
tags: [lora, unsloth, contract-clause-extraction, legal, apprentice]
---

# Apprentice --- Gemma 4 E4B LoRA (contract clause extraction)

LoRA adapter fine-tuned on 140 golden examples (CUAD v1, Contract Understanding
Atticus Dataset, 510 real commercial contracts from SEC EDGAR, expert-annotated
by lawyers, CC-BY-4.0) to extract document_name, parties, agreement_date,
governing_law, anti_assignment from a contract excerpt as JSON.

## Results (60 held-out rows, field-level F1)

Fill in from this task's README after publishing this run's printed numbers.

## Training

LoRA r=16, alpha 16, 3 epochs, lr 2e-4, batch 2 x grad-accum 4, Unsloth 4-bit,
Colab GPU. Train/eval split: seed 42, 140/60 from 200 sampled rows -- identical
split across every model this task is fine-tuned on.

## Usage

Load with PEFT on top of `google/gemma-4-E4B-it`, or serve via vLLM with `--enable-lora`.
Caveat: evaluated on 60 rows with exact-string field matching -- re-validate
before production use.
"""

with open("apprentice-gemma4-e4b-lora-cuad/README.md", "w") as f:
    f.write(model_card)

model.push_to_hub(REPO_ID, private=False)
tokenizer.push_to_hub(REPO_ID, private=False)

from huggingface_hub import upload_file
upload_file(path_or_fileobj="apprentice-gemma4-e4b-lora-cuad/README.md", path_in_repo="README.md", repo_id=REPO_ID)
print(f"pushed -> https://huggingface.co/{REPO_ID}")

## Gemma 4 E2B

`google/gemma-4-E2B-it` -- 2B effective params, the smaller sibling of E4B; same multimodal Processor caveat applies

In [ ]:
# 15. Load Gemma 4 E2B in 4-bit + LoRA
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    "google/gemma-4-E2B-it",
    max_seq_length=4096,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

In [ ]:
# 16. Pre-finetune baseline of the raw SLM on the held-out split
raw_slm_score = evaluate_slm(model, tokenizer, devset)
print(f"Gemma 4 E2B raw (no fine-tune): {raw_slm_score:.2f}")

In [ ]:
# 17. Build chat-format training data and fine-tune (SFT)
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

train_ds = Dataset.from_list([to_text(tokenizer, r) for r in trainset])

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=5,
        output_dir="outputs-apprentice-gemma4-e2b-lora-cuad",
        report_to="none",
    ),
)
trainer.train()

In [ ]:
# 18. Post-finetune eval on the SAME held-out split: the comparison that matters
finetuned_score = evaluate_slm(model, tokenizer, devset)
results["Gemma 4 E2B"] = {"raw": raw_slm_score, "finetuned": finetuned_score}

glm52_baseline = 34.67  # optimized blocked -- OpenCode gateway instability, see README
gpt54mini_baseline = 34.00
gpt54mini_optimized = 36.33

print("=" * 52)
print(f"glm-5.2 baseline prompt      : {glm52_baseline:.2f}")
print(f"gpt-5.4-mini baseline prompt : {gpt54mini_baseline:.2f}")
print(f"gpt-5.4-mini GEPA-optimized  : {gpt54mini_optimized:.2f}")
print(f"Gemma 4 E2B raw               : {raw_slm_score:.2f}")
print(f"Gemma 4 E2B fine-tuned        : {finetuned_score:.2f}")
print("=" * 52)
print("GO criterion: fine-tuned SLM >= gpt-5.4-mini baseline prompt score (34.00)")

In [ ]:
# 19. Persist the adapter (plan: artifacts go to a private HF Hub repo, not Drive),
# then free GPU memory before the next model
model.save_pretrained("apprentice-gemma4-e2b-lora-cuad")
tokenizer.save_pretrained("apprentice-gemma4-e2b-lora-cuad")
print("saved -> apprentice-gemma4-e2b-lora-cuad/")

del model, tokenizer, trainer
free_gpu_memory()

In [ ]:
# 19b. Write the model card and push to your HF Hub repo (same card style as
# the published receipt-extraction adapters).
# Fill in the printed numbers from cell 18 before pushing -- never push a card
# with a number that was not actually printed by this run.
from huggingface_hub import login

login()  # paste a HF token with write access when prompted

HF_USERNAME = "YOUR_HF_USERNAME"
REPO_ID = f"{HF_USERNAME}/apprentice-gemma4-e2b-lora-cuad"

model_card = """---
datasets: [theatticusproject/cuad-qa]
base_model: google/gemma-4-E2B-it
library_name: peft
license: apache-2.0
tags: [lora, unsloth, contract-clause-extraction, legal, apprentice]
---

# Apprentice --- Gemma 4 E2B LoRA (contract clause extraction)

LoRA adapter fine-tuned on 140 golden examples (CUAD v1, Contract Understanding
Atticus Dataset, 510 real commercial contracts from SEC EDGAR, expert-annotated
by lawyers, CC-BY-4.0) to extract document_name, parties, agreement_date,
governing_law, anti_assignment from a contract excerpt as JSON.

## Results (60 held-out rows, field-level F1)

Fill in from this task's README after publishing this run's printed numbers.

## Training

LoRA r=16, alpha 16, 3 epochs, lr 2e-4, batch 2 x grad-accum 4, Unsloth 4-bit,
Colab GPU. Train/eval split: seed 42, 140/60 from 200 sampled rows -- identical
split across every model this task is fine-tuned on.

## Usage

Load with PEFT on top of `google/gemma-4-E2B-it`, or serve via vLLM with `--enable-lora`.
Caveat: evaluated on 60 rows with exact-string field matching -- re-validate
before production use.
"""

with open("apprentice-gemma4-e2b-lora-cuad/README.md", "w") as f:
    f.write(model_card)

model.push_to_hub(REPO_ID, private=False)
tokenizer.push_to_hub(REPO_ID, private=False)

from huggingface_hub import upload_file
upload_file(path_or_fileobj="apprentice-gemma4-e2b-lora-cuad/README.md", path_in_repo="README.md", repo_id=REPO_ID)
print(f"pushed -> https://huggingface.co/{REPO_ID}")

## All three, side by side

In [ ]:
# 20. Every model's raw + fine-tuned score, printed together
print("=" * 60)
for name, scores in results.items():
    print(f"{name:16s} raw: {scores['raw']:6.2f}   fine-tuned: {scores['finetuned']:6.2f}")
print("=" * 60)

## After this run

1. Publish every raw/fine-tuned number exactly as printed -- this repo never carries a number that was not measured.
2. Add rows to this task's README results table for all three models.
3. Gemma 4 ships Apache-2.0 (no custom Terms-of-Use restrictions), same as Qwen3.5-4B -- safe to publish every fine-tuned adapter the same way.
4. If APIs error: Unsloth moves fast. Cross-check against the current official Unsloth notebook for whichever model failed.